# 06: Lab - Data Scraping

This practice notebook provides additional preparation for Assignment 6. 

## Setup
* Import packages: `requests`, `BeautifulSoup` (from `bs4`), `pandas` (as `pd`), `seaborn` (as `sns`)
* Set your Seaborn theme's style, context, and any runtime configurations you wish

In [ ]:
#Import packages
import requests
from bs4 import BeautifulSoup
import pandas as pd
import seaborn as sns

#Set Seaborn theme
sns.set_theme(style='darkgrid', context='notebook')

## Scraping Exercises

### 1. Scrape 2024 electricity usage data from the EIA 
Let's scrape data the US Energy Administration's State Electricity Archive:
<https://www.eia.gov/electricity/state/>

1. Navigate to the page. Note the year for which these data are provided.
2. Use the Selector Gadget tool to scrape the following values into variables: 
    * State name
    * Average Retail price
    * Net summer capacity
    * Net generation
    * Total retail sales
3. Compile the data into a dataframe
4. Examine the dataframe

**Recall that the BeautifulSoup scraping workflow is as follows**: 
1. Retrieve to the website contents using the `requests.get()` function.
1. Parse the contents into "soup" using BeautifulSoup.
1. Extract specific elements--or *element sets*--in the web site via their node IDs, found using Selector Gadget
    - Convert into lists using list comprehension<br>*Hint:* `[items.get_text(strip=True) for items in the_website.select('<nodeID>')]`
1. Wrangle values into a dataframe
    - Note that scraped values are always returned as strings; you'll need to manually convert these to numeric values (`float` or `int`)
    - Also note that with numbers scraped with a comma separator (e.g. `1,000`), the comma needs to be removed;  `str.replace(',','')` is helpful here
1. Examine the column information for your dataframe.
1. [Optional] Construct a regression plot (`regplot`) showing the relationship of total retail sales to net generation of electricity. 

In [ ]:
# 1.1 Fetch and parse the website
response = requests.get('https://www.eia.gov/electricity/state/')
the_website = BeautifulSoup(response.text,'html.parser')

# 1.2 Locate elements and read their text attributes into variables
the_states = [items.get_text(strip=True) for items in the_website.select('td:nth-child(1)')]
the_prices = [items.get_text(strip=True) for items in the_website.select('td:nth-child(2)')]
the_capacities = [items.get_text(strip=True) for items in the_website.select('td:nth-child(3)')]
net_generations = [items.get_text(strip=True) for items in the_website.select('td:nth-child(4)')]
total_retails = [items.get_text(strip=True) for items in the_website.select('td:nth-child(5)')]
  
# 1.3 Construct a dataframe from the values
df_usage = pd.DataFrame({
    'state':the_states,
    'price_USD':the_prices,
    'capacity_MW':the_capacities,
    'net_generation_MWh':net_generations,
    'total_retail_MWh':total_retails
}).iloc[:-1]

# 1.4 Wrangle
df_usage['price_USD'] = df_usage['price_USD'].astype('float')
df_usage['capacity_MW'] = df_usage['capacity_MW'].str.replace(',','').astype('int')
df_usage['net_generation_MWh'] = df_usage['net_generation_MWh'].str.replace(',','').astype('int')
df_usage['total_retail_MWh'] = df_usage['total_retail_MWh'].str.replace(',','').astype('int')

# 1.5 Examine
df_usage.info()

# 1.6. Plot
sns.regplot(
    data=df_usage,
    x='net_generation_MWh',
    y='total_retail_MWh',
).set(
    title= 'Electricity Profile: 2024',
    xlabel='Net Generation (MWh)',
    ylabel='Total Retail Sales (MWh)'
    );

### 2. Repeat, for a different year
* Copy and paste your code in the code chunk below, then modify it to fetch the same data for 2023
    - Note that the URL takes a slightly different format than 2024, but the HTML tags remain the same. 

In [ ]:
#Copy and paste code from above; then modify to scrape data for 2023
response = requests.get('https://www.eia.gov/electricity/state/archive/2023/')
the_website = BeautifulSoup(response.text,'html.parser')

the_states = [items.get_text(strip=True) for items in the_website.select('td:nth-child(1)')]
the_prices = [items.get_text(strip=True) for items in the_website.select('td:nth-child(2)')]
the_capacities = [items.get_text(strip=True) for items in the_website.select('td:nth-child(3)')]
net_generations = [items.get_text(strip=True) for items in the_website.select('td:nth-child(4)')]
total_retails = [items.get_text(strip=True) for items in the_website.select('td:nth-child(5)')]
  
df_usage = pd.DataFrame({
    'state':the_states,
    'price_USD':the_prices,
    'capacity_MW':the_capacities,
    'net_generation_MWh':net_generations,
    'total_retail_MWh':total_retails
}).iloc[:-1]

df_usage['price_USD'] = df_usage['price_USD'].astype('float')
df_usage['capacity_MW'] = df_usage['capacity_MW'].str.replace(',','').astype('int')
df_usage['net_generation_MWh'] = df_usage['net_generation_MWh'].str.replace(',','').astype('int')
df_usage['total_retail_MWh'] = df_usage['total_retail_MWh'].str.replace(',','').astype('int')

df_usage.info()

sns.regplot(
    data=df_usage,
    x='net_generation_MWh',
    y='total_retail_MWh',
).set(
    title= 'Electricity Profile: 2023',
    xlabel='Net Generation (MWh)',
    ylabel='Total Retail Sales (MWh)'
    )

### 3. Create a scraping *function* to accept a year as the input
* Note, you'll need to add an `if` statement to check whether the year is 2024 or not as the URL changes for years before 2024
* Add a `year` column to the dataframe, setting its value to the year supplied when the function is called. 

In [ ]:
# 3. Create a scrape function
def scrape_it(the_year):

    # 3.1 Form the base url based on whether the year is 2024 or not
    if the_year == 2024:
        base_url = 'https://www.eia.gov/electricity/state/'
    else:
        base_url = f'https://www.eia.gov/electricity/state/archive/{the_year}/'

    # 3.2 Fetch & parse the website
    response = requests.get(base_url)
    the_website = BeautifulSoup(response.text,'html.parser')

    # 3.3 Extract the data to list variables
    the_states = [items.get_text(strip=True) for items in the_website.select('td:nth-child(1)')]
    the_prices = [items.get_text(strip=True) for items in the_website.select('td:nth-child(2)')]
    the_capacities = [items.get_text(strip=True) for items in the_website.select('td:nth-child(3)')]
    net_generations = [items.get_text(strip=True) for items in the_website.select('td:nth-child(4)')]
    total_retails = [items.get_text(strip=True) for items in the_website.select('td:nth-child(5)')]

    # 3.4 Construct a dataframe from the extracted list variables
    df_usage = pd.DataFrame({
        'state':the_states,
        'price_USD':the_prices,
        'capacity_MW':the_capacities,
        'net_generation_MWh':net_generations,
        'total_retail_MWh':total_retails,
    }).iloc[:-1]

    # 4.5 Wrangle as needed
    df_usage['price_USD'] = df_usage['price_USD'].astype('float')
    df_usage['capacity_MW'] = df_usage['capacity_MW'].str.replace(',','').astype('int')
    df_usage['net_generation_MWh'] = df_usage['net_generation_MWh'].str.replace(',','').astype('int')
    df_usage['total_retail_MWh'] = df_usage['total_retail_MWh'].str.replace(',','').astype('int')
    df_usage['year'] = the_year

    # 4.6 Return the dataframe
    return df_usage

### 4. Use the function to scrape data for 2017

In [ ]:
#Use the function to scrape data for 2017
df_2017 = scrape_it(2017)
df_2017.head()

### 5. Use the function to scrape data for 2017-2024
* Construct dataframes for each year
* Concatenate the dataframes into a single dataframe
* Plot electricity prices for Virginia, North Carolina, and South Carolina for the span 2017-2024

In [ ]:
# 5.1 Extract and concatenate the dataframes
df_range = pd.concat(
    [scrape_it(the_year) for the_year in range(2017,2025)]
)

In [ ]:
# 5.2 Plot electricity cost for VA, NC, SC
sns.relplot(
    data=df_range[df_range['state'].isin(['Virginia','North Carolina','South Carolina'])],
    x='year',
    y='price_USD',
    hue='state',
    kind='line',
    height=3,
    aspect=4
).set(
    title='Electricity cost: VA, NC, SC',
    xlabel='Year',
    ylabel='Price (USD)'
);